In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'llm.py': '"""Greedy chat generation shared by the screening runs: vLLM (LLM_BACKEND=vllm, default) or transformers (LLM_BACKEND=hf).\n\n    m = LLM("Qwen/Qwen3-4B-Instruct-2507", max_len=16384, tp=1)\n    texts = m.gen([[{"role": "system", "content": ...}, {"role": "user", "content": ...}], ...], max_tokens=512)\n\nOne process loads one repo; trying the next candidate (fp16 -> AWQ, vllm -> hf) is multirun.py\'s job, in a new process.\nPrompts are tokenized here (chat template, enable_thinking off unless think=True) and passed as token ids, so no double BOS.\nA prompt longer than max_len - max_tokens comes back as None and is counted in m.overflow (never crashes the batch).\nText after </think> is kept when a model still thinks. m.ntok_in / m.ntok_out count tokens of every call.\n"""\nimport os\nimport time\n\n# ChatML, for tokenizers that ship no chat template (vilm/vinallama-2.7b-chat): the format its model card describes\nCHATML = ("{% for m in messages %}{{ \'<|im_start|>\' + m[\'role\'] + \'\\n\' + m[\'content\'] + \'<|im_end|>\\n\' }}{% endfor %}"\n          "{% if add_generation_prompt %}{{ \'<|im_start|>assistant\\n\' }}{% endif %}")\n\n\nclass LLM:\n    def __init__(self, repo, max_len=16384, tp=1, think=False, **kw):\n        self.repo, self.max_len, self.think = repo, max_len, think\n        self.overflow = self.ntok_in = self.ntok_out = 0\n        self.backend = os.environ.get("LLM_BACKEND", "vllm")\n        t0 = time.time()\n        if self.backend == "vllm":\n            from vllm import LLM as V\n            self.eng = V(model=repo, dtype="float16", max_model_len=max_len, tensor_parallel_size=tp, seed=0,\n                         gpu_memory_utilization=float(os.environ.get("GPU_UTIL", "0.90")), trust_remote_code=True, **kw)\n            self.tok = self.eng.get_tokenizer()\n        else:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            import transformers.integrations.sdpa_attention as sd\n            if hasattr(sd, "use_gqa_in_sdpa"):  # T4: enable_gqa only runs in the math kernel (heads x L x L scores, OOM): repeat K/V instead\n                sd.use_gqa_in_sdpa = lambda *a, **k: False\n            dkw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"\n            self.tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)\n            cuda = torch.cuda.is_available()\n            load = lambda remote: AutoModelForCausalLM.from_pretrained(repo, **{dkw: torch.float16 if cuda else torch.float32},\n                                                                       device_map="auto" if cuda else None, trust_remote_code=remote).eval()\n            try:\n                self.model = load(True)\n            except ImportError as e:  # repo code written for an older transformers (Phi-4-mini: LossKwargs); the built-in class works\n                print(f"[llm] remote code failed ({e}); loading the built-in transformers class", flush=True)\n                self.model = load(False)\n        if not getattr(self.tok, "chat_template", None):\n            self.tok.chat_template = CHATML\n            print(f"[llm] {repo}: no chat template, using ChatML", flush=True)\n        print(f"[llm] {repo} loaded with {self.backend} in {time.time() - t0:.0f}s", flush=True)\n\n    def ids(self, msgs):\n        kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=self.think)\n        try:\n            text = self.tok.apply_chat_template(msgs, **kw)\n        except Exception:  # templates without a system role: fold it into the first user turn\n            if msgs[0]["role"] != "system":\n                raise\n            text = self.tok.apply_chat_template([dict(msgs[1], content=msgs[0]["content"] + "\\n\\n" + msgs[1]["content"])] + msgs[2:], **kw)\n        if os.environ.get("STRIP_EMPTY_THINK") == "1":  # models fine-tuned by PIPE-T1-04 train.py on Qwen3.5 (no empty think block)\n            text = text.replace("<think>\\n\\n</think>\\n\\n", "")\n        return self.tok(text, add_special_tokens=False).input_ids\n\n    def gen(self, batch, max_tokens):\n        ids = [self.ids(m) for m in batch]\n        ok = [i for i, x in enumerate(ids) if len(x) <= self.max_len - max_tokens]\n        self.overflow += len(ids) - len(ok)\n        self.ntok_in += sum(len(ids[i]) for i in ok)\n        out = [None] * len(batch)\n        if self.backend == "vllm":\n            from vllm import SamplingParams\n            sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)\n            res = self.eng.generate([{"prompt_token_ids": ids[i]} for i in ok], sp, use_tqdm=False)\n            for i, r in zip(ok, res):\n                out[i] = r.outputs[0].text\n                self.ntok_out += len(r.outputs[0].token_ids)\n        else:\n            import torch\n            bs = int(os.environ.get("HF_BATCH", "4"))\n            pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id\n            dev = next(self.model.parameters()).device\n            order = sorted(ok, key=lambda i: len(ids[i]))\n            for k in range(0, len(order), bs):\n                part = order[k:k + bs]\n                n = max(len(ids[i]) for i in part)\n                x = torch.tensor([[pad] * (n - len(ids[i])) + ids[i] for i in part], device=dev)\n                mask = torch.tensor([[0] * (n - len(ids[i])) + [1] * len(ids[i]) for i in part], device=dev)\n                with torch.no_grad():\n                    y = self.model.generate(input_ids=x, attention_mask=mask, max_new_tokens=max_tokens, do_sample=False, pad_token_id=pad)\n                for i, row in zip(part, y[:, n:].tolist()):\n                    row = [t for t in row if t != pad]\n                    self.ntok_out += len(row)\n                    out[i] = self.tok.decode(row, skip_special_tokens=True)\n        return [o.split("</think>")[-1].strip() if o is not None else None for o in out]\n',
    'multirun.py': '"""Run one worker process per model over the GPUs of a Kaggle T4 x2 box.\n\n    ONLY=tag,tag python -u multirun.py jobs.json worker.py [-- extra worker args]      (ONLY empty or unset: every job)\n\njobs.json: [{"tag": "q3-4b", "tp": 1, "env": {"HARNESS": "4"}, "cands": [{"repo": "...", "backend": "vllm", ...}, ...], ...}, ...]\n(env: extra environment of that job\'s worker processes)\nEach candidate runs as `python -u worker.py --job \'<job + candidate as json>\' extra...` in its own process (a failed vLLM\nengine leaves a dirty process); exit code 0 ends the job, anything else tries the next candidate (fp16 -> AWQ, vllm -> hf).\ntp 2 jobs run one at a time on both GPUs first; tp 1 jobs then run two at a time, one per GPU. No GPU: one at a time.\nAfter a candidate its Hugging Face cache folder is deleted (Kaggle disk) unless the next candidate uses the same repo, or KEEP_HF=1.\nEvery worker line is echoed with a [tag] prefix; a summary of which candidate won is written to multirun_summary.json.\n"""\nimport json\nimport os\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nLOCK = threading.Lock()\n\n\ndef log(msg):\n    with LOCK:\n        print(msg, flush=True)\n\n\ndef ngpu():\n    try:\n        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())\n    except FileNotFoundError:\n        return 0\n\n\ndef run_job(job, worker, extra, gpus):\n    for k, cand in enumerate(job["cands"]):\n        spec = dict(job, **cand, cand_index=k)\n        spec.pop("cands")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", LLM_BACKEND=cand.get("backend", "vllm"), **job.get("env", {}))\n        t0 = time.time()\n        log(f"[{job[\'tag\']}] start candidate {k}: {cand[\'repo\']} ({env[\'LLM_BACKEND\']}) on GPU \'{gpus}\'")\n        p = subprocess.Popen([sys.executable, "-u", worker, "--job", json.dumps(spec, ensure_ascii=False)] + extra,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        tail = []\n        for line in p.stdout:\n            line = line.rstrip()\n            tail = (tail + [line])[-15:]\n            if "it/s]" not in line and "s/it]" not in line:  # drop tqdm bars\n                log(f"[{job[\'tag\']}] {line}")\n        rc = p.wait()\n        nxt = job["cands"][k + 1]["repo"] if k + 1 < len(job["cands"]) else None\n        if os.environ.get("KEEP_HF") != "1" and (rc == 0 or nxt != cand["repo"]):  # keep the weights for a retry of the same repo\n            hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"\n            shutil.rmtree(hub / ("models--" + cand["repo"].replace("/", "--")), ignore_errors=True)\n        log(f"[{job[\'tag\']}] candidate {k} rc {rc} after {time.time() - t0:.0f}s")\n        if rc == 0:\n            return dict(tag=job["tag"], repo=cand["repo"], cand_index=k, ok=True, sec=round(time.time() - t0))\n    return dict(tag=job["tag"], ok=False, tail=tail)\n\n\ndef main():\n    argv = sys.argv[1:]\n    extra = argv[argv.index("--") + 1:] if "--" in argv else []\n    argv = argv[:argv.index("--")] if "--" in argv else argv\n    jobs = json.load(open(argv[0], encoding="utf-8"))\n    worker = argv[1]\n    if os.environ.get("ONLY"):\n        keep = os.environ["ONLY"].split(",")\n        jobs = [j for j in jobs if j["tag"] in keep]\n    n = ngpu()\n    log(f"[multirun] {len(jobs)} jobs, {n} GPU(s): {[j[\'tag\'] for j in jobs]}")\n    done = []\n    if n < 2:\n        done = [run_job(j, worker, extra, "0" if n else "") for j in jobs]\n    else:\n        done = [run_job(j, worker, extra, "0,1") for j in jobs if j.get("tp", 1) > 1]\n        q = queue.Queue()\n        for j in jobs:\n            if j.get("tp", 1) == 1:\n                q.put(j)\n\n        def drain(g):\n            while True:\n                try:\n                    j = q.get_nowait()\n                except queue.Empty:\n                    return\n                done.append(run_job(j, worker, extra, g))\n\n        ts = [threading.Thread(target=drain, args=(str(g),)) for g in range(2)]\n        [t.start() for t in ts]\n        [t.join() for t in ts]\n    json.dump(done, open("multirun_summary.json", "w"), indent=1)\n    log("[multirun] summary: " + json.dumps([(d["tag"], d.get("repo"), d["ok"]) for d in done]))\n\n\nif __name__ == "__main__":\n    main()\n',
    'irlib.py': '"""IR of one quantitative environmental figure + deterministic comparison (GreenScan AG-1, GS-Bench v1).\n\nA model may fill the IR; every number, unit, period, scope and boundary check is done here, so the\nverdict always points at an arithmetic step (invariants 4-6 of ARCHITECTURE_INVARIANTS.md):\n  - one dimension differs (metric, value kind, status, period, scope, boundary, unit) -> N, never C;\n  - a target never proves a result;\n  - values match within the precision each side was published to.\nPure stdlib so the same file runs on Kaggle and in the product.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import asdict, dataclass, field\n\nMETRICS = {\n    "ghg_emissions": "phát thải khí nhà kính",\n    "energy_consumption": "tiêu thụ năng lượng",\n    "electricity_consumption": "tiêu thụ điện",\n    "renewable_energy": "năng lượng tái tạo (sản xuất hoặc sử dụng)",\n    "fuel_consumption": "tiêu thụ nhiên liệu (than, dầu, khí, sinh khối)",\n    "water_use": "nước khai thác / sử dụng / tiêu thụ",\n    "wastewater": "nước thải phát sinh / xử lý / tuần hoàn",\n    "waste_generated": "chất thải phát sinh",\n    "waste_recycled": "chất thải tái chế / tái sử dụng / thu hồi",\n    "hazardous_waste": "chất thải nguy hại",\n    "recycled_input": "nguyên liệu tái chế đưa vào sản xuất",\n    "air_pollutants": "bụi, SO2, NOx, khí thải khác",\n    "land_trees": "trồng cây, diện tích xanh, rừng",\n    "other_env": "chỉ số môi trường khác",\n}\nVALUE_KINDS = ("absolute", "intensity", "share", "change", "count")\nSTATUSES = ("actual", "target")\nSCOPES = ("s1", "s2", "s3", "s1+s2", "s1+s2+s3")\nBOUNDARIES = ("group", "parent", "facility", "unspecified")\n\nMULTIPLIERS = {"nghìn": 1e3, "ngàn": 1e3, "ngàn tỷ": 1e12, "nghìn tỷ": 1e12, "triệu": 1e6, "tỷ": 1e9, "tỉ": 1e9,\n               "thousand": 1e3, "million": 1e6, "billion": 1e9, "": 1.0}\n\n# unit text -> (dimension, factor to the dimension\'s base unit)\n_UNITS = [\n    (r"(kt|nghìn tấn) ?co2", ("ghg_t", 1e3)), (r"kg ?co2", ("ghg_t", 1e-3)), (r"(t|tấn) ?co2", ("ghg_t", 1.0)),\n    (r"twh", ("energy_kwh", 1e9)), (r"gwh", ("energy_kwh", 1e6)), (r"mwh", ("energy_kwh", 1e3)), (r"kwh", ("energy_kwh", 1.0)),\n    (r"toe", ("energy_kwh", 11_630.0)), (r"\\btj\\b", ("energy_kwh", 277_777.8)), (r"\\bgj\\b", ("energy_kwh", 277.7778)),\n    (r"\\bmwp?\\b", ("power_mw", 1.0)), (r"\\bkwp?\\b", ("power_mw", 1e-3)),\n    (r"m3|m³|khối", ("volume_m3", 1.0)), (r"lít|\\bl\\b", ("volume_m3", 1e-3)),\n    (r"tấn|\\bt\\b|tons?|tonnes?", ("mass_t", 1.0)), (r"\\bkg\\b", ("mass_t", 1e-3)),\n    (r"%|phần trăm", ("percent", 1.0)), (r"\\bha\\b|hecta|héc", ("area_ha", 1.0)), (r"cây|trees?", ("count_tree", 1.0)),\n]\n\n\ndef unit_dimension(unit_text: str) -> tuple[str, float]:\n    """Dimension and factor of a unit; an intensity keeps its denominator (\'ghg_t/tấn thép\')."""\n    text = (unit_text or "").strip().lower()\n    num, _, den = text.partition("/")\n    dim, factor = next(((d, f) for pattern, (d, f) in _UNITS if re.search(pattern, num)), (num or "none", 1.0))\n    if den:\n        dim += "/" + re.sub(r"\\s+", " ", den.strip())\n    return dim, factor\n\n\ndef parse_number(text: str) -> tuple[float, float]:\n    """Value of a written number and half its last written digit (the published precision).\n\n    Vietnamese writes 22.540.603 and 23,5; some reports use 1,234.5. A separator followed by exactly\n    three digits is read as a thousands separator, any other as the decimal mark.\n    """\n    s = re.sub(r"\\s", "", text or "")\n    if not re.fullmatch(r"[\\d.,]+", s) or not re.search(r"\\d", s):\n        raise ValueError(f"not a number: {text!r}")\n    if "." in s and "," in s:\n        dec = "," if s.rfind(",") > s.rfind(".") else "."\n        integer, decimals = s.replace("." if dec == "," else ",", "").split(dec)\n    elif s.count(".") > 1 or s.count(",") > 1:\n        integer, decimals = re.sub(r"[.,]", "", s), ""\n    elif "." in s or "," in s:\n        integer, decimals = re.split(r"[.,]", s)\n        if len(decimals) == 3:\n            integer, decimals = integer + decimals, ""\n    else:\n        integer, decimals = s, ""\n    value = float(f"{integer or 0}.{decimals or 0}")\n    return value, 0.5 * 10 ** (-len(decimals))\n\n\n_NUM_RE = re.compile(r"(?<!\\d)\\d[\\d.,]*\\d|(?<![\\d.,])\\d")\n_MULT_WORDS = sorted((w for w in MULTIPLIERS if w), key=len, reverse=True)\n_YEAR_RE = re.compile(r"(?<!\\d)(?:19|20)\\d\\d(?!\\d)")\n\n\ndef _multiplier_at(text: str) -> str:\n    """Multiplier word that opens text (\'triệu tCO2e\' -> \'triệu\'), \'\' when none."""\n    text = text.lstrip().lower()\n    return next((w for w in _MULT_WORDS if re.match(re.escape(w) + r"(?!\\w)(?!\\s+lệ(?!\\w))", text)), "")  # not "tỷ lệ" (ratio)\n\n\ndef ground(fig: dict, sentence: str) -> dict | None:\n    """Hold a reader\'s figure to what the sentence writes (AG-1 error attribution, 31/10).\n    - number, multiplier and unit packed into value_text (\'23,5 triệu tCO2e\') are split;\n    - the number must be written in the sentence (same value; the sentence\'s own spelling is kept), else None;\n    - the multiplier is the one the sentence writes right after that number (\'\' when it writes none):\n      a reader adding \'nghìn\' to \'10.730 m3\' turned 11 of Qwen3-32B\'s verdicts wrong;\n    - period and baseline year must be years written in the sentence, else null.\n    """\n    value = str(fig.get("value_text") or "").strip()\n    m = _NUM_RE.search(value)\n    if not m:\n        return None\n    out = {**fig, "value_text": m.group(0)}\n    rest = value[m.end():].strip()\n    if rest:\n        mult = _multiplier_at(rest)\n        if mult and not (fig.get("multiplier_text") or "").strip():\n            out["multiplier_text"] = mult\n        rest = rest[len(mult):].strip() if mult else rest\n        if rest and not (fig.get("unit_text") or "").strip():\n            out["unit_text"] = rest\n    try:\n        want = parse_number(out["value_text"])[0]\n    except ValueError:\n        return None\n    hits = []\n    for t in _NUM_RE.finditer(sentence):\n        try:\n            if parse_number(t.group(0))[0] == want:\n                hits.append(t)\n        except ValueError:\n            pass\n    hits = [t for t in hits if t.group(0) == out["value_text"]] or hits\n    if not hits:\n        return None\n    out["value_text"] = hits[0].group(0)\n    written = {_multiplier_at(sentence[t.end():t.end() + 16]) for t in hits}\n    mult = (out.get("multiplier_text") or "").strip().lower()\n    if mult not in written:\n        if len(written) != 1:\n            return None\n        out["multiplier_text"] = written.pop()\n    years = {int(y) for y in _YEAR_RE.findall(sentence)}\n    for name in ("period", "baseline_year"):\n        v = out.get(name)\n        if v not in (None, "", "null") and (not str(v).isdigit() or int(v) not in years):\n            out[name] = None\n    return out\n\n\n@dataclass\nclass IR:\n    metric: str\n    value_text: str\n    unit_text: str = ""\n    multiplier_text: str = ""\n    value_kind: str = "absolute"\n    change_direction: str | None = None  # decrease | increase, for value_kind == change\n    baseline_year: int | None = None\n    period: int | None = None\n    status: str = "actual"\n    scope: str | None = None\n    boundary: str = "unspecified"\n    metric_text: str = ""\n    subject_text: str = ""\n    extra: dict = field(default_factory=dict)\n\n    @classmethod\n    def from_dict(cls, d: dict) -> IR:\n        known = {k: d.get(k) for k in cls.__dataclass_fields__ if k in d and k != "extra"}\n        ir = cls(**known)\n        ir.boundary = ir.boundary or "unspecified"\n        ir.status = ir.status or "actual"\n        ir.value_kind = ir.value_kind or "absolute"\n        for name in ("period", "baseline_year"):\n            v = getattr(ir, name)\n            setattr(ir, name, int(v) if v not in (None, "", "null") and str(v).isdigit() else None)\n        return ir\n\n    def as_dict(self) -> dict:\n        return asdict(self)\n\n    def quantity(self) -> tuple[float, float, str]:\n        """Value and tolerance in the dimension\'s base unit, and the dimension."""\n        value, half = parse_number(self.value_text)\n        mult = MULTIPLIERS.get((self.multiplier_text or "").strip().lower(), 1.0)\n        dim, factor = unit_dimension(self.unit_text)\n        return value * mult * factor, half * mult * factor, dim\n\n\n_KIND = {"count": "absolute"}  # v1.2: a count is an absolute figure; the unit dimension (cây, sự cố …) still tells them apart\n\n\ndef comparable(claim: IR, evidence: IR) -> tuple[bool, str]:\n    """Same metric, value kind, status, period, scope, boundary and unit dimension (invariant 4)."""\n    checks = [\n        ("chỉ số", claim.metric, evidence.metric),\n        ("loại giá trị", _KIND.get(claim.value_kind, claim.value_kind), _KIND.get(evidence.value_kind, evidence.value_kind)),\n        ("thực hiện / mục tiêu", claim.status, evidence.status),\n        ("kỳ", claim.period, evidence.period),\n        ("phạm vi phát thải", claim.scope, evidence.scope),\n    ]\n    if claim.value_kind == "change":\n        checks.append(("năm gốc", claim.baseline_year, evidence.baseline_year))\n    if "unspecified" not in (claim.boundary, evidence.boundary):\n        checks.append(("ranh giới tổ chức", claim.boundary, evidence.boundary))\n    for name, a, b in checks:\n        if a != b:\n            return False, f"khác {name}: {a} / {b}"\n    try:\n        dim_c, dim_e = claim.quantity()[2], evidence.quantity()[2]\n    except ValueError as exc:\n        return False, f"không đọc được số: {exc}"\n    if dim_c != dim_e:\n        return False, f"khác đơn vị: {dim_c} / {dim_e}"\n    return True, "so được"\n\n\ndef compare(claim: IR, evidence: IR) -> tuple[str, str]:\n    """S (khớp), C (mâu thuẫn) or N (không so được), with the arithmetic or the reason."""\n    ok, why = comparable(claim, evidence)\n    if not ok:\n        return "N", why\n    if claim.value_kind == "change" and claim.change_direction != evidence.change_direction:\n        return "C", f"ngược chiều: {claim.change_direction} / {evidence.change_direction}"\n    vc, tc, dim = claim.quantity()\n    ve, te, _ = evidence.quantity()\n    gap = abs(vc - ve)\n    label = "S" if gap <= tc + te else "C"\n    return label, f"|{vc:g} − {ve:g}| = {gap:g} {dim}; dung sai {tc + te:g}"\n\n\ndef written_in(ir: IR, text: str) -> bool:\n    """The text writes this figure\'s number with the same multiplier (whatever a reader extracted from it)."""\n    fig = ground({"value_text": ir.value_text, "multiplier_text": ir.multiplier_text}, text)\n    return fig is not None and (fig.get("multiplier_text") or "").lower() == (ir.multiplier_text or "").lower()\n\n\ndef decide(claim: IR, evidence: list[IR], evidence_text: str | None = None) -> tuple[str, list[tuple[str, str]]]:\n    """Verdict over every evidence figure: S if some comparable figure matches and none conflicts,\n    C if comparable figures only conflict, N otherwise (nothing comparable, or mixed: a person reads it).\n    v1.2: with the evidence text, a C that rests on a value gap becomes N when the text writes the claim\'s own\n    number and multiplier: the reader missed that figure (both false C of Qwen3-4B on held-out run D)."""\n    steps = [compare(claim, e) for e in evidence]\n    labels = {label for label, _ in steps if label != "N"}\n    label = labels.pop() if len(labels) == 1 else "N"\n    direction = any(w.startswith("ngược chiều") for _, w in steps)\n    if label == "C" and evidence_text is not None and not direction and written_in(claim, evidence_text):\n        steps.append(("N", "số của tuyên bố có trong câu bằng chứng nhưng chưa được đọc ra: chuyển người"))\n        label = "N"\n    return label, steps\n',
    'ag1_worker.py': '"""AG-1 worker: one local LLM reads every claim and evidence text of GS-Bench v1 into IR figures; irlib decides.\n\n    python -u ag1_worker.py --job \'{"tag": "q3-4b-inst", "repo": "Qwen/Qwen3-4B-Instruct-2507", "tp": 1, "max_len": 4096}\'\n\nRuns inside a Kaggle notebook built by build_nb.py (files embedded next to it: llm.py from ZAIC Experiment/common,\nirlib.py, gs_bench_v1.jsonl). Greedy, thinking off. The model only fills the IR; S / C / N comes from irlib.decide\n(arithmetic by code, invariant 5). Output: out/<tag>_texts.jsonl (raw + parsed figures per unique text) and\nout/<tag>_cases.jsonl (verdict per case), plus a one-line summary in the log.\nEnv: LIMIT=n (first n cases, smoke), MAX_TOKENS (default 768).\n"""\nimport argparse, hashlib, json, os, re, time\nfrom pathlib import Path\n\nfrom irlib import IR, METRICS, SCOPES, VALUE_KINDS, decide, ground\n\nPROMPT = """Bạn trích số liệu môi trường từ MỘT câu trong báo cáo của doanh nghiệp Việt Nam.\nVới MỖI con số mô tả một chỉ số môi trường (không lấy số năm, số điều luật, số thứ tự, số tiền), trả về một đối tượng:\n- "metric": một mã trong {metrics}\n- "metric_text": cụm từ gọi tên chỉ số, chép NGUYÊN VĂN từ câu\n- "value_text": con số, chép NGUYÊN VĂN từ câu (giữ dấu chấm, phẩy), không kèm chữ\n- "multiplier_text": "nghìn", "triệu", "tỷ" nếu câu viết chữ đó ngay sau số, ngược lại ""\n- "unit_text": đơn vị nguyên văn (vd "tCO2e", "kWh", "m3", "tấn", "%", "tCO2e/tấn sản phẩm"); "" nếu không có\n- "value_kind": một trong {kinds} (share = tỷ lệ phần trăm của một tổng; change = mức tăng/giảm so với kỳ trước hoặc năm gốc; intensity = trên một đơn vị sản phẩm/doanh thu)\n- "change_direction": "decrease" | "increase" | null (chỉ khi value_kind = change)\n- "baseline_year": năm gốc nếu là change và câu có nêu, ngược lại null\n- "period": năm của số liệu nếu câu NÊU RÕ; null nếu câu không nêu\n- "status": "actual" (đã thực hiện) | "target" (mục tiêu, kế hoạch, cam kết)\n- "scope": một trong {scopes} nếu là phát thải khí nhà kính và câu nêu phạm vi, ngược lại null\n- "boundary": "group" (tập đoàn, hợp nhất) | "parent" (công ty mẹ) | "facility" (nhà máy, cơ sở cụ thể) | "unspecified"\nCâu không có số liệu môi trường thì trả về danh sách rỗng.\n\nCÂU: {sentence}\n\nChỉ trả lời đúng một đối tượng JSON: {{"figures": [ ... ]}}"""\n\n\ndef parse(text):\n    """First JSON object in the reply; None when there is none."""\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth = 0\n        for i in range(start, len(text)):\n            depth += {"{": 1, "}": -1}.get(text[i], 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef to_irs(figures, sentence=None):\n    """Readable figures; with the sentence, each is first held to what the sentence writes (irlib.ground)."""\n    out = []\n    for f in figures or []:\n        if isinstance(f, dict) and f.get("value_text") not in (None, ""):\n            if sentence is not None and (f := ground(f, sentence)) is None:\n                continue\n            try:\n                ir = IR.from_dict({**f, "value_text": str(f["value_text"])})\n                ir.quantity()  # unreadable number -> dropped\n                out.append(ir)\n            except (TypeError, ValueError):\n                pass\n    return out\n\n\ndef verdict(claim_irs, ev_irs, ev_text=None):\n    if not claim_irs:\n        return "N", ["không đọc được số liệu trong tuyên bố"]\n    labels, why = [], []\n    for c in claim_irs:\n        label, steps = decide(c, ev_irs, ev_text)\n        labels.append(label)\n        why.append(f"{c.value_text} {c.unit_text}: {label} " + "; ".join(s[1] for s in steps)[:300])\n    if "C" in labels:\n        return "C", why\n    return ("S" if all(x == "S" for x in labels) else "N"), why\n\n\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument("--job")\n    job = json.loads(ap.parse_args().job)\n    tag = job["tag"]\n    cases = [json.loads(line) for line in open(os.environ.get("GS_BENCH", "gs_bench_v1.jsonl"), encoding="utf-8")]\n    if os.environ.get("LIMIT"):\n        cases = cases[: int(os.environ["LIMIT"])]\n    texts = {}\n    for c in cases:\n        for t in (c["claim"], c["evidence"]):\n            texts.setdefault(hashlib.sha1(t.encode()).hexdigest()[:16], t)\n    for path in filter(None, os.environ.get("EXTRA_INPUT", "").split(",")):  # more sentences to read, no cases (second reader)\n        for r in map(json.loads, open(path, encoding="utf-8")):\n            texts.setdefault(hashlib.sha1(r["text"].encode()).hexdigest()[:16], r["text"])\n    print(f"[{tag}] {len(cases)} cases, {len(texts)} unique texts, repo {job[\'repo\']}", flush=True)\n    from llm import LLM\n    t0 = time.time()\n    m = LLM(job["repo"], max_len=job.get("max_len", 4096), tp=job.get("tp", 1), **job.get("kw", {}))\n    load_s = time.time() - t0\n    keys = list(texts)\n    prompts = [[{"role": "user", "content": PROMPT.format(metrics=list(METRICS), kinds=list(VALUE_KINDS), scopes=list(SCOPES),\n                                                         sentence=texts[k])}] for k in keys]\n    t1 = time.time()\n    raw = []\n    for i in range(0, len(prompts), 64):\n        raw += m.gen(prompts[i:i + 64], max_tokens=int(os.environ.get("MAX_TOKENS", "768")))\n        print(f"[{tag}] generated {len(raw)}/{len(prompts)}", flush=True)\n    gen_s = time.time() - t1\n    Path("out").mkdir(exist_ok=True)\n    irs, bad = {}, 0\n    with open(f"out/{tag}_texts.jsonl", "w", encoding="utf-8") as fh:\n        for k, r in zip(keys, raw):\n            obj = parse(r)\n            bad += obj is None\n            figs = (obj or {}).get("figures") if isinstance((obj or {}).get("figures"), list) else []\n            irs[k] = to_irs(figs, None if os.environ.get("GROUND") == "0" else texts[k])\n            fh.write(json.dumps({"key": k, "text": texts[k], "raw": r, "figures": [x.as_dict() for x in irs[k]]}, ensure_ascii=False) + "\\n")\n    right = 0\n    with open(f"out/{tag}_cases.jsonl", "w", encoding="utf-8") as fh:\n        for c in cases:\n            kc, ke = (hashlib.sha1(t.encode()).hexdigest()[:16] for t in (c["claim"], c["evidence"]))\n            label, why = verdict(irs[kc], irs[ke], None if os.environ.get("GROUND") == "0" else c["evidence"])\n            right += label in c["labels"]\n            fh.write(json.dumps({"id": c["id"], "label": label, "why": why}, ensure_ascii=False) + "\\n")\n    summary = {"tag": tag, "repo": job["repo"], "cases": len(cases), "right": right, "unparsed_texts": bad,\n               "load_s": round(load_s), "gen_s": round(gen_s), "ntok_in": getattr(m, "ntok_in", None), "ntok_out": getattr(m, "ntok_out", None)}\n    json.dump(summary, open(f"out/{tag}_summary.json", "w"), ensure_ascii=False)\n    print(f"[{tag}] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'gs_heldout_v12.jsonl': '{"id": "bh-256e9fb0b2610f5e-0-S0", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 300 kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["S"]}\n{"id": "bh-256e9fb0b2610f5e-0-C1", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 375 kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["C"]}\n{"id": "bh-256e9fb0b2610f5e-0-C2", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 300 nghìn kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["C"]}\n{"id": "bh-256e9fb0b2610f5e-0-N2", "claim": "Công ty đặt mục tiêu tiết kiệm trung bình 300-1.000 kw điện đạt 300 kw vào năm 2030.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["N"]}\n{"id": "bh-256e9fb0b2610f5e-0-N5", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 300 kw/tấn sản phẩm.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["N"]}\n{"id": "bh-256e9fb0b2610f5e-0-N6", "claim": "Lượng điện từ năng lượng tái tạo của Công ty là 300 kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["N"]}\n{"id": "bh-86c0150d1bdb2265-0-S0", "claim": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh của Công ty là 02 Đơn vị.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["S"]}\n{"id": "bh-86c0150d1bdb2265-0-C1", "claim": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh của Công ty là 7 Đơn vị.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["C"]}\n{"id": "bh-86c0150d1bdb2265-0-C2", "claim": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh của Công ty là 02 nghìn Đơn vị.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["C"]}\n{"id": "bh-86c0150d1bdb2265-0-N2", "claim": "Công ty đặt mục tiêu Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh đạt 02 Đơn vị vào năm 2030.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["N"]}\n{"id": "bh-2924cd23d07ec79f-0-S0", "claim": "Năng suất của Công ty tăng 10% so với năm 2020.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["S"]}\n{"id": "bh-2924cd23d07ec79f-0-C1", "claim": "Năng suất của Công ty tăng 25% so với năm 2020.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["C"]}\n{"id": "bh-2924cd23d07ec79f-0-C3", "claim": "Năng suất của Công ty giảm 10% so với năm 2020.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["C"]}\n{"id": "bh-2924cd23d07ec79f-0-N2", "claim": "Công ty đặt mục tiêu Năng suất tăng 10% vào năm 2030.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["N"]}\n{"id": "bh-12d6bfbb4289bee1-0-S0", "claim": "Thiệt hại của biến đổi khí hậu lên nền kinh tế của Công ty đạt 1,5 % GDP.", "evidence": "tế của nước ta trong thời gian qua do ô nhiễm môi trường gây ra tối thiểu từ 1,5% - 3% GDP Những thiệt hại của biến đổi khí hậu lên nền kinh tế của quốc gia gây ra rủi ro cho hoạt động kinh doanh của Traphaco xét ở khía cạnh thu nhập của người dân giảm và các ưu tên dành cho chăm sóc sức khỏe có thể được thay thế cho những nguy cơ khác cấp bách hơn của khách hàng nhằm phục vụ cho các mục têu ngắn hạn.", "labels": ["S"]}\n{"id": "bh-12d6bfbb4289bee1-0-C1", "claim": "Thiệt hại của biến đổi khí hậu lên nền kinh tế của Công ty đạt 16,5 % GDP.", "evidence": "tế của nước ta trong thời gian qua do ô nhiễm môi trường gây ra tối thiểu từ 1,5% - 3% GDP Những thiệt hại của biến đổi khí hậu lên nền kinh tế của quốc gia gây ra rủi ro cho hoạt động kinh doanh của Traphaco xét ở khía cạnh thu nhập của người dân giảm và các ưu tên dành cho chăm sóc sức khỏe có thể được thay thế cho những nguy cơ khác cấp bách hơn của khách hàng nhằm phục vụ cho các mục têu ngắn hạn.", "labels": ["C"]}\n{"id": "bh-12d6bfbb4289bee1-0-N2", "claim": "Công ty đặt mục tiêu thiệt hại của biến đổi khí hậu lên nền kinh tế đạt 1,5 % GDP vào năm 2030.", "evidence": "tế của nước ta trong thời gian qua do ô nhiễm môi trường gây ra tối thiểu từ 1,5% - 3% GDP Những thiệt hại của biến đổi khí hậu lên nền kinh tế của quốc gia gây ra rủi ro cho hoạt động kinh doanh của Traphaco xét ở khía cạnh thu nhập của người dân giảm và các ưu tên dành cho chăm sóc sức khỏe có thể được thay thế cho những nguy cơ khác cấp bách hơn của khách hàng nhằm phục vụ cho các mục têu ngắn hạn.", "labels": ["N"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-S0", "claim": "Năm 2025, cường độ phát thải khí nhà kính của Công ty đạt 0,03%.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["S"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-C1", "claim": "Năm 2025, cường độ phát thải khí nhà kính của Công ty đạt 15,03%.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["C"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-N1", "claim": "Năm 2024, cường độ phát thải khí nhà kính của Công ty đạt 0,03%.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["N"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-N2", "claim": "Công ty đặt mục tiêu cường độ phát thải khí nhà kính đạt 0,03% vào năm 2030.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["N"]}\n{"id": "bh-42bb04afc41b5a1c-1-S0", "claim": "Năm 2022, tiết giảm được giảm 12% của Công ty đạt 12%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["S"]}\n{"id": "bh-42bb04afc41b5a1c-1-C1", "claim": "Năm 2022, tiết giảm được giảm 12% của Công ty đạt 27%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["C"]}\n{"id": "bh-42bb04afc41b5a1c-1-N1", "claim": "Năm 2021, tiết giảm được giảm 12% của Công ty đạt 12%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["N"]}\n{"id": "bh-42bb04afc41b5a1c-1-N2", "claim": "Công ty đặt mục tiêu tiết giảm được giảm 12% đạt 12% vào năm 2030.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["N"]}\n{"id": "bh-42bb04afc41b5a1c-1-N6", "claim": "Năm 2022, lượng nước thải của Công ty đạt 12%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["N"]}\n{"id": "bh-489f9d772e830f0e-0-S0", "claim": "Diện tích của Công ty là 950 ha.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["S"]}\n{"id": "bh-489f9d772e830f0e-0-C1", "claim": "Diện tích của Công ty là 1.188 ha.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["C"]}\n{"id": "bh-489f9d772e830f0e-0-C2", "claim": "Diện tích của Công ty là 950 nghìn ha.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["C"]}\n{"id": "bh-489f9d772e830f0e-0-N2", "claim": "Công ty đặt mục tiêu diện tích đạt 950 ha vào năm 2030.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["N"]}\n{"id": "bh-489f9d772e830f0e-0-N5", "claim": "Diện tích của Công ty là 950 ha/tấn sản phẩm.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["N"]}\n{"id": "bh-2abb39872ba73557-0-S0", "claim": "Thu hồi được của Công ty là 7.000 viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["S"]}\n{"id": "bh-2abb39872ba73557-0-C1", "claim": "Thu hồi được của Công ty là 4.900 viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["C"]}\n{"id": "bh-2abb39872ba73557-0-C2", "claim": "Thu hồi được của Công ty là 7.000 nghìn viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["C"]}\n{"id": "bh-2abb39872ba73557-0-N2", "claim": "Công ty đặt mục tiêu thu hồi được đạt 7.000 viên pin cũ vào năm 2030.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["N"]}\n{"id": "bh-2abb39872ba73557-0-N5", "claim": "Thu hồi được của Công ty là 7.000 viên pin cũ/tấn sản phẩm.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["N"]}\n{"id": "bh-2abb39872ba73557-0-N6", "claim": "Lượng chất thải phát sinh của Công ty là 7.000 viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["N"]}\n{"id": "bh-752eb0e07be3dea9-0-S0", "claim": "Công ty đặt mục tiêu cây xanh đạt 1 cây.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["S"]}\n{"id": "bh-752eb0e07be3dea9-0-C1", "claim": "Công ty đặt mục tiêu cây xanh đạt 6 cây.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["C"]}\n{"id": "bh-752eb0e07be3dea9-0-N2", "claim": "Cây xanh của Công ty là 1 cây.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["N"]}\n{"id": "bh-752eb0e07be3dea9-0-N5", "claim": "Công ty đặt mục tiêu cây xanh đạt 1 cây/tấn sản phẩm.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["N"]}\n{"id": "bh-e4a22e9b25091836-0-S0", "claim": "Tỷ trọng vốn chủ sở hữu của Công ty đạt 67%.", "evidence": "Tỷ trọng nợ phải trả và nguồn vốn chủ sở hữu không có biến động nhiều trong các năm gần đây đặc »» Tài sản cố định/Tổng Tài sản »» Tài sản lưu động/Tổng Tài sản »» Nợ phải trả/Tổng nguồn vốn »» Nguồn vốn chủ sở hữu/Tổng nguồn vốn Biểu đồ cấu trúc tài sản - nguồn vốn (%) Nguồn vốn chủ sở hữu/ biệt tỷ trọng vốn chủ sở hữu luôn chiếm trên 67% tổng nguồn vốn thể hiện cơ cấu tài chính Công ty mang tính lành mạnh và bền vững.", "labels": ["S"]}\n{"id": "bh-e4a22e9b25091836-0-C1", "claim": "Tỷ trọng vốn chủ sở hữu của Công ty đạt 82%.", "evidence": "Tỷ trọng nợ phải trả và nguồn vốn chủ sở hữu không có biến động nhiều trong các năm gần đây đặc »» Tài sản cố định/Tổng Tài sản »» Tài sản lưu động/Tổng Tài sản »» Nợ phải trả/Tổng nguồn vốn »» Nguồn vốn chủ sở hữu/Tổng nguồn vốn Biểu đồ cấu trúc tài sản - nguồn vốn (%) Nguồn vốn chủ sở hữu/ biệt tỷ trọng vốn chủ sở hữu luôn chiếm trên 67% tổng nguồn vốn thể hiện cơ cấu tài chính Công ty mang tính lành mạnh và bền vững.", "labels": ["C"]}\n{"id": "bh-e4a22e9b25091836-0-N2", "claim": "Công ty đặt mục tiêu tỷ trọng vốn chủ sở hữu đạt 67% vào năm 2030.", "evidence": "Tỷ trọng nợ phải trả và nguồn vốn chủ sở hữu không có biến động nhiều trong các năm gần đây đặc »» Tài sản cố định/Tổng Tài sản »» Tài sản lưu động/Tổng Tài sản »» Nợ phải trả/Tổng nguồn vốn »» Nguồn vốn chủ sở hữu/Tổng nguồn vốn Biểu đồ cấu trúc tài sản - nguồn vốn (%) Nguồn vốn chủ sở hữu/ biệt tỷ trọng vốn chủ sở hữu luôn chiếm trên 67% tổng nguồn vốn thể hiện cơ cấu tài chính Công ty mang tính lành mạnh và bền vững.", "labels": ["N"]}\n{"id": "bh-079acf7a91109441-0-S0", "claim": "Rừng cao su của Công ty là 9 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["S"]}\n{"id": "bh-079acf7a91109441-0-C1", "claim": "Rừng cao su của Công ty là 13 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"]}\n{"id": "bh-079acf7a91109441-0-C2", "claim": "Rừng cao su của Công ty là 9 triệu ha.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"]}\n{"id": "bh-079acf7a91109441-0-N2", "claim": "Công ty đặt mục tiêu rừng cao su đạt 9 ngàn ha vào năm 2030.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"]}\n{"id": "bh-079acf7a91109441-0-N5", "claim": "Rừng cao su của Công ty là 9 ngàn ha/tấn sản phẩm.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"]}\n{"id": "bh-cdea1d4e020ecf9e-0-S0", "claim": "Năm 2025, thiệt hại về tài sản của Công ty là 119,9 nghìn tỷ đồng.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["S"]}\n{"id": "bh-cdea1d4e020ecf9e-0-C1", "claim": "Năm 2025, thiệt hại về tài sản của Công ty là 95,9 nghìn tỷ đồng.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["C"]}\n{"id": "bh-cdea1d4e020ecf9e-0-N1", "claim": "Năm 2024, thiệt hại về tài sản của Công ty là 119,9 nghìn tỷ đồng.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["N"]}\n{"id": "bh-cdea1d4e020ecf9e-0-N2", "claim": "Công ty đặt mục tiêu thiệt hại về tài sản đạt 119,9 nghìn tỷ đồng vào năm 2030.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["N"]}\n{"id": "bh-cdea1d4e020ecf9e-0-N5", "claim": "Năm 2025, thiệt hại về tài sản của Công ty là 119,9 nghìn tỷ đồng/tấn sản phẩm.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["N"]}\n{"id": "bh-24ec31c88923203e-0-S0", "claim": "56% doanh thu của Công ty đạt 56%.", "evidence": "Hết năm 2012 tổng doanh thu của công ty đạt 1.477 tỷ đồng trong đó doanh thu các sản phẩm từ dược liệu của công ty đã đạt 56% doanh thu (tăng 28% so với năm 2011), những kết quả này là những minh chứng cho thấy hiệu quả của chiến lược sức khỏe xanh của TRAPHACO, đồng thời cũng khẳng định quan điểm phát triển bền vững của công ty: “Vì môi trường xanh, vì sức khỏe cộng đồng, vì sự phồn vinh của đất nước”.", "labels": ["S"]}\n{"id": "bh-24ec31c88923203e-0-C1", "claim": "56% doanh thu của Công ty đạt 71%.", "evidence": "Hết năm 2012 tổng doanh thu của công ty đạt 1.477 tỷ đồng trong đó doanh thu các sản phẩm từ dược liệu của công ty đã đạt 56% doanh thu (tăng 28% so với năm 2011), những kết quả này là những minh chứng cho thấy hiệu quả của chiến lược sức khỏe xanh của TRAPHACO, đồng thời cũng khẳng định quan điểm phát triển bền vững của công ty: “Vì môi trường xanh, vì sức khỏe cộng đồng, vì sự phồn vinh của đất nước”.", "labels": ["C"]}\n{"id": "bh-24ec31c88923203e-0-N2", "claim": "Công ty đặt mục tiêu 56% doanh thu đạt 56% vào năm 2030.", "evidence": "Hết năm 2012 tổng doanh thu của công ty đạt 1.477 tỷ đồng trong đó doanh thu các sản phẩm từ dược liệu của công ty đã đạt 56% doanh thu (tăng 28% so với năm 2011), những kết quả này là những minh chứng cho thấy hiệu quả của chiến lược sức khỏe xanh của TRAPHACO, đồng thời cũng khẳng định quan điểm phát triển bền vững của công ty: “Vì môi trường xanh, vì sức khỏe cộng đồng, vì sự phồn vinh của đất nước”.", "labels": ["N"]}\n{"id": "bh-7e4c135fa8258de0-0-S0", "claim": "Điện sản xuất của Công ty là 860.512 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"]}\n{"id": "bh-7e4c135fa8258de0-0-S1", "claim": "Điện sản xuất của Công ty là 860,5 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"]}\n{"id": "bh-7e4c135fa8258de0-0-C1", "claim": "Điện sản xuất của Công ty là 602.358 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"]}\n{"id": "bh-7e4c135fa8258de0-0-C2", "claim": "Điện sản xuất của Công ty là 860.512 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"]}\n{"id": "bh-7e4c135fa8258de0-0-N2", "claim": "Công ty đặt mục tiêu Điện sản xuất đạt 860.512 KW vào năm 2030.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"]}\n{"id": "bh-7e4c135fa8258de0-0-N5", "claim": "Điện sản xuất của Công ty là 860.512 KW/tấn sản phẩm.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"]}\n{"id": "bh-7e4c135fa8258de0-0-N6", "claim": "Lượng điện từ năng lượng tái tạo của Công ty là 860.512 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"]}\n{"id": "bh-7e4c135fa8258de0-1-S0", "claim": "Điện văn phòng của Công ty là 241.728 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"]}\n{"id": "bh-7e4c135fa8258de0-1-S1", "claim": "Điện văn phòng của Công ty là 241,7 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"]}\n{"id": "bh-7e4c135fa8258de0-1-C1", "claim": "Điện văn phòng của Công ty là 193.382 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"]}\n{"id": "bh-7e4c135fa8258de0-1-C2", "claim": "Điện văn phòng của Công ty là 241.728 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"]}\n{"id": "bh-7e4c135fa8258de0-1-N2", "claim": "Công ty đặt mục tiêu Điện văn phòng đạt 241.728 KW vào năm 2030.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"]}\n{"id": "bh-7e4c135fa8258de0-1-N5", "claim": "Điện văn phòng của Công ty là 241.728 KW/tấn sản phẩm.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"]}\n{"id": "bh-7e4c135fa8258de0-1-N6", "claim": "Lượng điện từ năng lượng tái tạo của Công ty là 241.728 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"]}\n{"id": "bh-2b222a6ddb74a7f1-0-S0", "claim": "Rừng cao su của Công ty là 9 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["S"]}\n{"id": "bh-2b222a6ddb74a7f1-0-C1", "claim": "Rừng cao su của Công ty là 14 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"]}\n{"id": "bh-2b222a6ddb74a7f1-0-C2", "claim": "Rừng cao su của Công ty là 9 triệu ha.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"]}\n{"id": "bh-2b222a6ddb74a7f1-0-N2", "claim": "Công ty đặt mục tiêu rừng cao su đạt 9 ngàn ha vào năm 2030.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"]}\n{"id": "bh-2b222a6ddb74a7f1-0-N5", "claim": "Rừng cao su của Công ty là 9 ngàn ha/tấn sản phẩm.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-0-S0", "claim": "Năm 2022, chi phí điện năng của Công ty giảm 7,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"]}\n{"id": "bh-aae28ea7d360fa29-0-C1", "claim": "Năm 2022, chi phí điện năng của Công ty giảm 22,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-0-C3", "claim": "Năm 2022, chi phí điện năng của Công ty tăng 7,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-0-N2", "claim": "Công ty đặt mục tiêu chi phí điện năng giảm 7,6% vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-0-N6", "claim": "Năm 2022, lượng điện từ năng lượng tái tạo của Công ty giảm 7,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-1-S0", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 2.760 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"]}\n{"id": "bh-aae28ea7d360fa29-1-C1", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 3.864 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-1-C2", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 2.760 nghìn kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-1-N1", "claim": "Năm 2020, Khí gas phục vụ nấu nướng của Công ty là 2.760 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-1-N2", "claim": "Công ty đặt mục tiêu Khí gas phục vụ nấu nướng đạt 2.760 kg vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-1-N5", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 2.760 kg/tấn sản phẩm.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-2-S0", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"]}\n{"id": "bh-aae28ea7d360fa29-2-C1", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 1.069 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-2-C2", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 nghìn kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-2-N1", "claim": "Năm 2020, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-2-N2", "claim": "Công ty đặt mục tiêu Khí gas phục vụ nghiên cứu sản phẩm đạt 855 kg vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-2-N5", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 kg/tấn sản phẩm.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-4-S0", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 100 lít.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"]}\n{"id": "bh-aae28ea7d360fa29-4-C1", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 80 lít.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-4-C2", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 100 nghìn lít.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"]}\n{"id": "bh-aae28ea7d360fa29-4-N2", "claim": "Công ty đặt mục tiêu Tổng lượng dầu DO chạy phục vụ máy phát điện đạt 100 lít vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-aae28ea7d360fa29-4-N5", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 100 lít/tấn sản phẩm.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"]}\n{"id": "bh-bbd4e51bc7042cbc-0-S0", "claim": "Tỉ trọng lớn nhất của Công ty đạt 83,8%.", "evidence": "Doanh thu hàng sản xuất là dòng hàng chiếm tỉ trọng lớn nhất 83,8% trong cơ cấu doanh thu của công ty, thể hiện sự phát triển bền vững của doanh nghiệp dược trong lĩnh vực sản xuất các mặt hàng thuốc đông dược và tân dược chất lượng cao nhằm chăm sóc và bảo vệ sức Bên cạnh việc tăng trưởng ở hàng sản xuất thì hàng khai thác cũng có mức tăng trưởng trên 54% so với cùng kỳ năm 2019.", "labels": ["S"]}\n{"id": "bh-bbd4e51bc7042cbc-0-C1", "claim": "Tỉ trọng lớn nhất của Công ty đạt 68,8%.", "evidence": "Doanh thu hàng sản xuất là dòng hàng chiếm tỉ trọng lớn nhất 83,8% trong cơ cấu doanh thu của công ty, thể hiện sự phát triển bền vững của doanh nghiệp dược trong lĩnh vực sản xuất các mặt hàng thuốc đông dược và tân dược chất lượng cao nhằm chăm sóc và bảo vệ sức Bên cạnh việc tăng trưởng ở hàng sản xuất thì hàng khai thác cũng có mức tăng trưởng trên 54% so với cùng kỳ năm 2019.", "labels": ["C"]}\n{"id": "bh-bbd4e51bc7042cbc-0-N2", "claim": "Công ty đặt mục tiêu tỉ trọng lớn nhất đạt 83,8% vào năm 2030.", "evidence": "Doanh thu hàng sản xuất là dòng hàng chiếm tỉ trọng lớn nhất 83,8% trong cơ cấu doanh thu của công ty, thể hiện sự phát triển bền vững của doanh nghiệp dược trong lĩnh vực sản xuất các mặt hàng thuốc đông dược và tân dược chất lượng cao nhằm chăm sóc và bảo vệ sức Bên cạnh việc tăng trưởng ở hàng sản xuất thì hàng khai thác cũng có mức tăng trưởng trên 54% so với cùng kỳ năm 2019.", "labels": ["N"]}\n{"id": "bh-bea6856b8ea1f475-0-S0", "claim": "Năm 2025, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty tăng 3 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["S"]}\n{"id": "bh-bea6856b8ea1f475-0-C1", "claim": "Năm 2025, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty tăng 8 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["C"]}\n{"id": "bh-bea6856b8ea1f475-0-C3", "claim": "Năm 2025, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty giảm 3 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["C"]}\n{"id": "bh-bea6856b8ea1f475-0-N1", "claim": "Năm 2024, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty tăng 3 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["N"]}\n{"id": "bh-bea6856b8ea1f475-0-N2", "claim": "Công ty đặt mục tiêu cải thiện 3 bậc trong bảng điểm đánh giá tăng 3 bậc vào năm 2030.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["N"]}\n{"id": "bh-3c5c5f73d7747fec-0-S0", "claim": "Hơn 4 triệu sản phẩm của Công ty là 4 triệu sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["S"]}\n{"id": "bh-3c5c5f73d7747fec-0-C1", "claim": "Hơn 4 triệu sản phẩm của Công ty là 9 triệu sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["C"]}\n{"id": "bh-3c5c5f73d7747fec-0-C2", "claim": "Hơn 4 triệu sản phẩm của Công ty là 4 tỷ sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["C"]}\n{"id": "bh-3c5c5f73d7747fec-0-N2", "claim": "Công ty đặt mục tiêu hơn 4 triệu sản phẩm đạt 4 triệu sản phẩm vào năm 2030.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["N"]}\n{"id": "bh-3c5c5f73d7747fec-0-N5", "claim": "Hơn 4 triệu sản phẩm của Công ty là 4 triệu sản phẩm/tấn sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["N"]}\n{"id": "bh-ac0dcc6352bdadad-0-S0", "claim": "Thị phần của Công ty đạt 70%.", "evidence": "“Được thành lập năm 2007, với tầm nhìn tiên phong trong sự nghiệp xanh hóa nguồn nhiên liệu tại Việt Nam, Công ty CP CNG Việt Nam hiện là đơn vị hàng đầu về cung cấp khí thiên nhiên nén (CNG – Compressed Natural Gas) và khí thiên nhiên hóa lỏng (LNG – Liquefied Natural Gas), chiếm lĩnh >70% thị phần và có đầy đủ cơ sở hạ tầng vận chuyển – phân phối trên khắp cả nước.", "labels": ["S"]}\n{"id": "bh-ac0dcc6352bdadad-0-C1", "claim": "Thị phần của Công ty đạt 85%.", "evidence": "“Được thành lập năm 2007, với tầm nhìn tiên phong trong sự nghiệp xanh hóa nguồn nhiên liệu tại Việt Nam, Công ty CP CNG Việt Nam hiện là đơn vị hàng đầu về cung cấp khí thiên nhiên nén (CNG – Compressed Natural Gas) và khí thiên nhiên hóa lỏng (LNG – Liquefied Natural Gas), chiếm lĩnh >70% thị phần và có đầy đủ cơ sở hạ tầng vận chuyển – phân phối trên khắp cả nước.", "labels": ["C"]}\n{"id": "bh-ac0dcc6352bdadad-0-N2", "claim": "Công ty đặt mục tiêu thị phần đạt 70% vào năm 2030.", "evidence": "“Được thành lập năm 2007, với tầm nhìn tiên phong trong sự nghiệp xanh hóa nguồn nhiên liệu tại Việt Nam, Công ty CP CNG Việt Nam hiện là đơn vị hàng đầu về cung cấp khí thiên nhiên nén (CNG – Compressed Natural Gas) và khí thiên nhiên hóa lỏng (LNG – Liquefied Natural Gas), chiếm lĩnh >70% thị phần và có đầy đủ cơ sở hạ tầng vận chuyển – phân phối trên khắp cả nước.", "labels": ["N"]}\n{"id": "bh-8be4715d7a95430c-0-S0", "claim": "Năm 2025, tiêu thụ năng lượng điện của Công ty đạt 35%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["S"]}\n{"id": "bh-8be4715d7a95430c-0-C1", "claim": "Năm 2025, tiêu thụ năng lượng điện của Công ty đạt 50%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["C"]}\n{"id": "bh-8be4715d7a95430c-0-N1", "claim": "Năm 2024, tiêu thụ năng lượng điện của Công ty đạt 35%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["N"]}\n{"id": "bh-8be4715d7a95430c-0-N2", "claim": "Công ty đặt mục tiêu tiêu thụ năng lượng điện đạt 35% vào năm 2030.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["N"]}\n{"id": "bh-8be4715d7a95430c-0-N6", "claim": "Năm 2025, lượng điện từ năng lượng tái tạo của Công ty đạt 35%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["N"]}\n{"id": "bh-de130211938fb92a-0-S0", "claim": "Tạo ra việc làm cho của Công ty là 2.300 nhân viên.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["S"]}\n{"id": "bh-de130211938fb92a-0-C1", "claim": "Tạo ra việc làm cho của Công ty là 1.840 nhân viên.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["C"]}\n{"id": "bh-de130211938fb92a-0-C2", "claim": "Tạo ra việc làm cho của Công ty là 2.300 nghìn nhân viên.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["C"]}\n{"id": "bh-de130211938fb92a-0-N2", "claim": "Công ty đặt mục tiêu Tạo ra việc làm cho đạt 2.300 nhân viên vào năm 2030.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["N"]}\n{"id": "bh-de130211938fb92a-0-N5", "claim": "Tạo ra việc làm cho của Công ty là 2.300 nhân viên/tấn sản phẩm.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["N"]}\n{"id": "bh-e3f000701dd57fce-0-S0", "claim": "Chiếm tỷ trọng của Công ty đạt 39,2%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"]}\n{"id": "bh-e3f000701dd57fce-0-C1", "claim": "Chiếm tỷ trọng của Công ty đạt 54,2%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"]}\n{"id": "bh-e3f000701dd57fce-0-N2", "claim": "Công ty đặt mục tiêu Chiếm tỷ trọng đạt 39,2% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"]}\n{"id": "bh-e3f000701dd57fce-1-S0", "claim": "Chiếm tỷ trọng của Công ty đạt 43,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"]}\n{"id": "bh-e3f000701dd57fce-1-C1", "claim": "Chiếm tỷ trọng của Công ty đạt 58,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"]}\n{"id": "bh-e3f000701dd57fce-1-N2", "claim": "Công ty đặt mục tiêu Chiếm tỷ trọng đạt 43,6% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"]}\n{"id": "bh-e3f000701dd57fce-2-S0", "claim": "Chiếm tỷ trọng của Công ty đạt 14,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"]}\n{"id": "bh-e3f000701dd57fce-2-C1", "claim": "Chiếm tỷ trọng của Công ty đạt 29,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"]}\n{"id": "bh-e3f000701dd57fce-2-N2", "claim": "Công ty đặt mục tiêu Chiếm tỷ trọng đạt 14,6% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"]}\n{"id": "bh-e3f000701dd57fce-3-S0", "claim": "Năm 2025, đóng góp \uf0f110,9% của Công ty tăng 10,9% so với năm 2024.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"]}\n{"id": "bh-e3f000701dd57fce-3-C1", "claim": "Năm 2025, đóng góp \uf0f110,9% của Công ty tăng 25,9% so với năm 2024.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"]}\n{"id": "bh-e3f000701dd57fce-3-C3", "claim": "Năm 2025, đóng góp \uf0f110,9% của Công ty giảm 10,9% so với năm 2024.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"]}\n{"id": "bh-e3f000701dd57fce-3-N2", "claim": "Công ty đặt mục tiêu đóng góp \uf0f110,9% tăng 10,9% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"]}\n{"id": "bh-e771363ec78bac11-0-S0", "claim": "Tổng giờ thực hiện của Công ty là 651 giờ.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["S"]}\n{"id": "bh-e771363ec78bac11-0-C1", "claim": "Tổng giờ thực hiện của Công ty là 814 giờ.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["C"]}\n{"id": "bh-e771363ec78bac11-0-C2", "claim": "Tổng giờ thực hiện của Công ty là 651 nghìn giờ.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["C"]}\n{"id": "bh-e771363ec78bac11-0-N2", "claim": "Công ty đặt mục tiêu tổng giờ thực hiện đạt 651 giờ vào năm 2030.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["N"]}\n{"id": "bh-e771363ec78bac11-0-N5", "claim": "Tổng giờ thực hiện của Công ty là 651 giờ/tấn sản phẩm.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["N"]}\n{"id": "bh-1c6744cbbe0ddbef-0-S0", "claim": "Chai nhựa tái chế của Công ty là 5 chai.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["S"]}\n{"id": "bh-1c6744cbbe0ddbef-0-C1", "claim": "Chai nhựa tái chế của Công ty là 10 chai.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["C"]}\n{"id": "bh-1c6744cbbe0ddbef-0-C2", "claim": "Chai nhựa tái chế của Công ty là 5 nghìn chai.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["C"]}\n{"id": "bh-1c6744cbbe0ddbef-0-N2", "claim": "Công ty đặt mục tiêu chai nhựa tái chế đạt 5 chai vào năm 2030.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["N"]}\n{"id": "bh-1c6744cbbe0ddbef-0-N5", "claim": "Chai nhựa tái chế của Công ty là 5 chai/tấn sản phẩm.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["N"]}\n{"id": "bh-4eaa023bef040348-0-S0", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["S"]}\n{"id": "bh-4eaa023bef040348-0-S1", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3,0 triệu tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["S"]}\n{"id": "bh-4eaa023bef040348-0-C1", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 2.400.000 tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["C"]}\n{"id": "bh-4eaa023bef040348-0-C2", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 nghìn tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["C"]}\n{"id": "bh-4eaa023bef040348-0-N1", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 tấn vào năm 2025.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["N"]}\n{"id": "bh-4eaa023bef040348-0-N2", "claim": "Kế hoạch than dự kiến giao nhận của Công ty là 3.000.000 tấn.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["N"]}\n{"id": "bh-4eaa023bef040348-0-N5", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 tấn/tấn sản phẩm vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["N"]}\n{"id": "bh-3958feef5779ca82-0-S0", "claim": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO của Công ty là 0 m3.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["S"]}\n{"id": "bh-3958feef5779ca82-0-C1", "claim": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO của Công ty là 5 m3.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["C"]}\n{"id": "bh-3958feef5779ca82-0-N2", "claim": "Công ty đặt mục tiêu TỔNG LƯỢNG NƯỚC ĐẦU VÀO đạt 0 m3 vào năm 2030.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["N"]}\n{"id": "bh-3958feef5779ca82-0-N5", "claim": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO của Công ty là 0 m3/tấn sản phẩm.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["N"]}\n{"id": "bh-3958feef5779ca82-0-N6", "claim": "Lượng nước thải của Công ty là 0 m3.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["N"]}\n{"id": "bh-d51355abee1d0e73-0-S0", "claim": "Thuốc trong nước chỉ chiếm của Công ty đạt 50%.", "evidence": "Doanh nghiệp Dược nước ngoài với công nghệ hiện đại, kinh nghiệm quốc tế, tiềm lực tài chính mạnh cùng với tâm lý chuộng hàng ngoại là những nguyên nhân mà thuốc trong nước chỉ chiếm gần 50% thị phần cả nước.", "labels": ["S"]}\n{"id": "bh-d51355abee1d0e73-0-C1", "claim": "Thuốc trong nước chỉ chiếm của Công ty đạt 65%.", "evidence": "Doanh nghiệp Dược nước ngoài với công nghệ hiện đại, kinh nghiệm quốc tế, tiềm lực tài chính mạnh cùng với tâm lý chuộng hàng ngoại là những nguyên nhân mà thuốc trong nước chỉ chiếm gần 50% thị phần cả nước.", "labels": ["C"]}\n{"id": "bh-d51355abee1d0e73-0-N2", "claim": "Công ty đặt mục tiêu thuốc trong nước chỉ chiếm đạt 50% vào năm 2030.", "evidence": "Doanh nghiệp Dược nước ngoài với công nghệ hiện đại, kinh nghiệm quốc tế, tiềm lực tài chính mạnh cùng với tâm lý chuộng hàng ngoại là những nguyên nhân mà thuốc trong nước chỉ chiếm gần 50% thị phần cả nước.", "labels": ["N"]}\n',
    'jobs_g.json': '[\n {\n  "tag": "q3-4b-inst",\n  "tp": 1,\n  "max_len": 4096,\n  "env": {\n   "GS_BENCH": "gs_heldout_v12.jsonl"\n  },\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-4B-Instruct-2507"\n   }\n  ]\n },\n {\n  "tag": "q3-8b",\n  "tp": 2,\n  "max_len": 4096,\n  "env": {\n   "GS_BENCH": "gs_heldout_v12.jsonl"\n  },\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-8B"\n   }\n  ]\n },\n {\n  "tag": "q3-14b-awq",\n  "tp": 2,\n  "max_len": 4096,\n  "env": {\n   "GS_BENCH": "gs_heldout_v12.jsonl"\n  },\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-14B-AWQ"\n   }\n  ]\n },\n {\n  "tag": "q3-32b-awq",\n  "tp": 2,\n  "max_len": 4096,\n  "env": {\n   "GS_BENCH": "gs_heldout_v12.jsonl"\n  },\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-32B-AWQ"\n   }\n  ]\n }\n]',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u multirun.py jobs_g.json ag1_worker.py', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u multirun.py jobs_g.json ag1_worker.py')


In [ ]:
print('[cell] run 2: ls -la out; cat out/*_summary.json', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('ls -la out; cat out/*_summary.json')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)